# Track C: The Search — Optimization and the Ratchet

**Plan C — Parallel Tracks**

This track covers automated parameter search. You will learn how the ratchet generates challengers, selects winners, extracts lessons, and narrows the search space across rungs.

> **Dashboard:** Use `00_dashboard.ipynb` to explore individual experiments manually, then see how the ratchet does it automatically.

In [ ]:
%matplotlib inline
import warnings, tempfile
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

from autoresearch_quantum.models import (
    ExperimentSpec, RungConfig, EvaluationMetrics,
    QualityWeights, CostWeights, ScoreConfig, SearchSpaceConfig,
    TierPolicyConfig, HardwareConfig, LessonFeedback, SearchRule,
)
from autoresearch_quantum.execution.local import LocalCheapExecutor
from autoresearch_quantum.search.challengers import (
    generate_neighbor_challengers, mutation_summary, GeneratedChallenger,
)
from autoresearch_quantum.search.strategies import (
    NeighborWalk, RandomCombo, LessonGuided, CompositeGenerator,
    default_composite, StrategyWeight,
)
from autoresearch_quantum.ratchet.runner import AutoresearchHarness
from autoresearch_quantum.persistence.store import ResearchStore
from autoresearch_quantum.config import load_rung_config
from autoresearch_quantum.lessons.extractor import extract_rung_lesson
from autoresearch_quantum.lessons.feedback import (
    extract_search_rules, narrow_search_space, build_lesson_feedback,
)
from autoresearch_quantum.execution.transfer import TransferEvaluator
from matplotlib.patches import Patch

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_c_track_c")
print("Learning tracker active.")

---
## 1. The Parameter Space

The rung1 config defines a discrete search space over circuit parameters.

In [ ]:
rung_config = load_rung_config("../../configs/rungs/rung1.yaml")

print(f"Rung: {rung_config.name}")
print(f"Objective: {rung_config.objective}")
print(f"\nSearch dimensions:")
total_combos = 1
for dim, values in rung_config.search_space.dimensions.items():
    print(f"  {dim:25s}: {values}")
    total_combos *= len(values)
print(f"\nTotal combinations: {total_combos}")
print(f"Max challengers per step: {rung_config.search_space.max_challengers_per_step}")
print(f"Step budget: {rung_config.step_budget}")
print(f"Patience: {rung_config.patience}")

In [ ]:
quiz(tracker, "q1_why_search",
    question="The parameter space is finite. Why not just try every combination?",
    options=[
        "The space is infinite",
        "Each evaluation costs time/compute; smart search finds good solutions faster",
        "Exhaustive search always finds worse solutions",
    ],
    correct=1, section="1. Parameter space", bloom="understand",
    explanation="Each evaluation requires noisy simulation (or hardware QPU time). Smart search finds good solutions in fewer evaluations.")

With {total} combinations, exhaustive search is feasible but slow. The ratchet is smarter: it starts from a good baseline and explores *neighborhoods*, focusing compute where it matters.

---
## 2. The Incumbent-Challenger Model

Think of it like a chess championship:
- The **incumbent** is the reigning champion (best configuration found so far)
- **Challengers** are generated by mutating the incumbent's parameters
- Each challenger is evaluated ("plays a match")
- If a challenger beats the incumbent by a sufficient margin, it takes the title

This is a form of **local search** — like hill climbing in a discrete parameter space.

In [ ]:
incumbent = rung_config.bootstrap_incumbent
print("Bootstrap incumbent:")
for field in ["seed_style", "encoder_style", "verification", "postselection",
              "ancilla_strategy", "optimization_level"]:
    print(f"  {field:25s}: {getattr(incumbent, field)}")

In [ ]:
quiz(tracker, "q2_incumbent",
    question="What is the bootstrap incumbent?",
    options=[
        "A randomly chosen starting point",
        "A hand-picked reasonable default that the ratchet tries to beat",
        "The theoretically optimal configuration",
    ],
    correct=1, section="2. Incumbent", bloom="remember",
    explanation="The bootstrap incumbent is a domain-expert guess. The ratchet guarantee: it never gets worse from here.")
checkpoint_summary(tracker, "2. Incumbent")

---
## 3. NeighborWalk: Single-Axis Perturbation

The simplest strategy: change **one parameter at a time** and try all alternatives.

In [ ]:
challengers = generate_neighbor_challengers(incumbent, rung_config.search_space)

print(f"Generated {len(challengers)} challengers (max {rung_config.search_space.max_challengers_per_step}):\n")
for i, c in enumerate(challengers):
    print(f"  {i+1:2d}. {c.mutation_note}")

In [ ]:
quiz(tracker, "q3_neighborwalk",
    question="NeighborWalk changes how many parameters per challenger?",
    options=["0", "Exactly 1", "Up to 3", "All of them"],
    correct=1, section="3. NeighborWalk", bloom="understand",
    explanation="One parameter at a time, trying all alternative values. Systematic but blind to parameter interactions.")

> **Key Insight:** NeighborWalk is exhaustive within single axes but never tests *combinations*. It is fast and deterministic — good for identifying which individual parameter matters most.

---
## 4. RandomCombo: Multi-Axis Perturbation

Change 1-3 parameters simultaneously.

In [ ]:
combo = RandomCombo(num_candidates=8, max_mutations=3)
combo_challengers = combo.generate(incumbent, rung_config.search_space, set())

print(f"Generated {len(combo_challengers)} random combo challengers:\n")
for i, c in enumerate(combo_challengers):
    # Count how many fields changed
    n_changes = sum(1 for f in incumbent.__dataclass_fields__
                    if getattr(incumbent, f) != getattr(c.spec, f))
    print(f"  {i+1:2d}. [{n_changes} changes] {c.mutation_note}")

In [ ]:
order(tracker, "q4_strategy_interactions",
    instruction="Rank strategies by ability to find multi-parameter interactions (worst to best):",
    items=["NeighborWalk", "RandomCombo"],
    correct_order=["NeighborWalk", "RandomCombo"],
    section="4. RandomCombo", bloom="analyze",
    explanation="NeighborWalk: 1 axis only, cannot find interactions. RandomCombo mutates multiple axes simultaneously.")
checkpoint_summary(tracker, "4. RandomCombo")

> **Key Insight:** RandomCombo can discover *interaction effects* — parameter combinations that are better or worse than the sum of individual effects. It introduces diversity but is less systematic.

---
## 5. Evaluating: Incumbent vs Challengers

Let us actually run the experiments and see scores.

In [ ]:
# Use fast settings
fast_rung = RungConfig(
    rung=1, name=rung_config.name, description=rung_config.description,
    objective=rung_config.objective, bootstrap_incumbent=incumbent,
    search_space=rung_config.search_space,
    tier_policy=TierPolicyConfig(
        cheap_margin=0.002, cheap_shots=256, cheap_repeats=1,
        expensive_shots=512, expensive_repeats=1,
        promote_top_k=2, enable_hardware=False,
    ),
    score=rung_config.score,
    step_budget=1, patience=1, hardware=HardwareConfig(),
)

executor = LocalCheapExecutor()
inc_result = executor.evaluate(incumbent, fast_rung)
print(f"Incumbent score: {inc_result.score:.4f}\n")

# Evaluate neighbor challengers
scores = {}
for c in challengers[:8]:
    result = executor.evaluate(c.spec, fast_rung)
    scores[c.mutation_note] = result.score
    marker = ">>>" if result.score > inc_result.score else "   "
    print(f"  {marker} {c.mutation_note[:50]:50s} score={result.score:.4f}")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
labels = ["INCUMBENT"] + [k[:35] for k in scores.keys()]
vals = [inc_result.score] + list(scores.values())
colors = ["#e74c3c"] + ["#2ecc71" if s > inc_result.score else "#bdc3c7" for s in scores.values()]

ax.barh(range(len(labels)), vals, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.axvline(x=inc_result.score, color="#e74c3c", linestyle="--", alpha=0.5)
ax.set_xlabel("Score")
ax.set_title("Incumbent vs Challengers")
ax.legend(handles=[Patch(color="#e74c3c", label="Incumbent"),
                   Patch(color="#2ecc71", label="Beats incumbent"),
                   Patch(color="#bdc3c7", label="Below incumbent")])
plt.tight_layout()
plt.show()

---
## 6. One Ratchet Step in Detail

Now let the harness orchestrate everything: generate challengers, evaluate, promote, select winner.

In [ ]:
store = ResearchStore(tempfile.mkdtemp())
harness = AutoresearchHarness(store)

step = harness.run_ratchet_step(fast_rung, allow_hardware=False)

print(f"Step index:          {step.step_index}")
print(f"Incumbent before:    {step.incumbent_before_id}")
print(f"Challengers tested:  {len(step.challengers_tested)}")
print(f"Promoted:            {len(step.promoted_challengers)}")
print(f"Winner:              {step.winner_id}")
print(f"Winning margin:      {step.winning_margin:+.4f}")
print(f"\nCheap-tier: {step.cheap_tier_justification}")
print(f"\nLesson: {step.distilled_lesson}")

In [ ]:
quiz(tracker, "q5_no_winner",
    question="What happens if no challenger beats the incumbent?",
    options=[
        "The harness picks the best challenger anyway",
        "The incumbent stays; the step is logged with zero improvement",
        "The harness doubles the number of challengers",
    ],
    correct=1, section="6. Ratchet step", bloom="understand",
    explanation="Ratchet guarantee: the incumbent never gets worse. No-improvement steps are still valuable data for lesson extraction.")

---
## 7. Running a Full Rung with Patience

A rung runs multiple steps. **Patience** stops early if no improvement is found.

In [ ]:
store2 = ResearchStore(tempfile.mkdtemp())
harness2 = AutoresearchHarness(store2)

multi_rung = RungConfig(
    rung=1, name="Search Demo", description="Full rung demo",
    objective="Find best config",
    bootstrap_incumbent=ExperimentSpec(
        rung=1, target_backend="fake_brisbane", noise_backend="fake_brisbane",
        shots=256, repeats=1,
    ),
    search_space=SearchSpaceConfig(
        dimensions={
            "verification": ["both", "z_only", "x_only"],
            "seed_style": ["h_p", "ry_rz", "u_magic"],
            "postselection": ["all_measured", "z_only", "none"],
        },
        max_challengers_per_step=6,
    ),
    tier_policy=TierPolicyConfig(
        cheap_margin=0.001, cheap_shots=256, cheap_repeats=1,
        promote_top_k=2, enable_hardware=False,
    ),
    score=rung_config.score,
    step_budget=3, patience=2,
    hardware=HardwareConfig(),
)

steps, lesson, feedback = harness2.run_rung(multi_rung, allow_hardware=False)

print(f"Steps completed: {len(steps)}")
for s in steps:
    improved = "IMPROVED" if s.winning_margin > 0 else "no change"
    print(f"  Step {s.step_index}: margin={s.winning_margin:+.4f}  ({improved})")

In [ ]:
quiz(tracker, "q6_patience",
    question="Patience=2 means the rung stops after 2 consecutive steps with no improvement. Why?",
    options=[
        "To save memory",
        "If 2 rounds of challengers all lose, the nearby parameter space is likely exhausted",
        "2 is always the optimal patience value",
    ],
    correct=1, section="7. Full rung", bloom="evaluate",
    explanation="Patience prevents wasting compute once the search has converged. The budget is better spent on the next rung.")
checkpoint_summary(tracker, "7. Full rung")

In [ ]:
# Score trajectory
experiments = store2.list_experiments(1)
exp_data = [(e["experiment_id"][:20], e["final_score"], e["role"]) for e in experiments]

fig, ax = plt.subplots(figsize=(12, 4))
x = range(len(exp_data))
colors = ["#e74c3c" if role == "incumbent" else "#3498db" for _, _, role in exp_data]
ax.bar(x, [s for _, s, _ in exp_data], color=colors)
ax.set_xlabel("Experiment")
ax.set_ylabel("Score")
ax.set_title("All Experiments in Rung")
ax.legend(handles=[Patch(color="#e74c3c", label="Incumbent"), Patch(color="#3498db", label="Challenger")])
plt.tight_layout()
plt.show()

---
## 8. Lesson Extraction

After a rung, the harness analyzes all experiments and extracts **lessons** — both human-readable narratives and machine-readable rules.

In [ ]:
print("=" * 60)
print(lesson.narrative)
print("=" * 60)

In [ ]:
reflect(tracker, "q7_lesson_quality",
    question="Read the lesson narrative. What actionable insight does it give? What would make it better?",
    section="8. Lessons", bloom="evaluate",
    model_answer="A good lesson names specific values that helped/hurt and explains WHY. Machine-readable SearchRules are often more actionable than the narrative.")

In [ ]:
print(f"\nMachine-readable rules ({len(feedback.rules)}):\n")
for rule in feedback.rules:
    print(f"  {rule.action.upper():7s} {rule.dimension}={rule.value}")
    print(f"          confidence={rule.confidence:.2f}  reason: {rule.reason}\n")

In [ ]:
quiz(tracker, "q8_fix_vs_avoid",
    question="'fix' vs 'avoid' rules: what's the difference?",
    options=[
        "'fix' locks a value permanently; 'avoid' removes a value from the search space",
        "'fix' repairs a bug; 'avoid' prevents a crash",
        "They are synonyms",
    ],
    correct=0, section="8. Lessons", bloom="remember",
    explanation="'fix': always use this value. 'avoid': never use this value. Both narrow the search space for future rungs.")
checkpoint_summary(tracker, "8. Lessons")

---
## 9. LessonGuided Strategy

Once we have rules, the **LessonGuided** strategy uses them to bias challenger generation toward promising regions.

In [ ]:
if feedback.rules:
    guided = LessonGuided(num_candidates=6)
    guided_challengers = guided.generate(
        incumbent, multi_rung.search_space, set(), [feedback]
    )
    print(f"Lesson-guided: {len(guided_challengers)} challengers")
    for c in guided_challengers:
        print(f"  {c.mutation_note}")
else:
    print("No rules extracted — try a larger step_budget for more data.")

---
## 10. Search Space Narrowing

Rules can **narrow** the search space: remove "avoid" values, lock "fix" values.

In [ ]:
print("BEFORE narrowing:")
for dim, vals in multi_rung.search_space.dimensions.items():
    print(f"  {dim}: {vals}")

narrowed = narrow_search_space(multi_rung.search_space, feedback.rules)

print("\nAFTER narrowing:")
for dim, vals in narrowed.dimensions.items():
    removed = set(multi_rung.search_space.dimensions[dim]) - set(vals)
    suffix = f"  (removed: {removed})" if removed else ""
    print(f"  {dim}: {vals}{suffix}")

In [ ]:
quiz(tracker, "q9_narrowing",
    question="What does search space narrowing accomplish?",
    options=[
        "It removes entire parameter dimensions",
        "It removes poorly-performing values, keeping the dimension with fewer options",
        "It adds new parameter values",
    ],
    correct=1, section="10. Narrowing", bloom="understand",
    explanation="Narrowing prunes bad values based on evidence. The dimension stays but with fewer options. A minimum is preserved to prevent overfitting.")

> **Key Insight:** Narrowing is "learning" — the machine prunes bad options based on evidence. This makes subsequent rungs faster and more focused.

---
## 11. Cross-Rung Propagation

A full **ratchet** chains multiple rungs. The winner from rung $N$ becomes the bootstrap for rung $N+1$.

In [ ]:
store3 = ResearchStore(tempfile.mkdtemp())
harness3 = AutoresearchHarness(store3)

rung1 = RungConfig(
    rung=1, name="Rung 1", description="Explore basics",
    objective="Find best seed and verification",
    bootstrap_incumbent=ExperimentSpec(
        rung=1, target_backend="fake_brisbane", noise_backend="fake_brisbane",
        shots=256, repeats=1,
    ),
    search_space=SearchSpaceConfig(
        dimensions={"verification": ["both", "z_only"], "seed_style": ["h_p", "ry_rz"]},
        max_challengers_per_step=4,
    ),
    tier_policy=TierPolicyConfig(cheap_margin=0.0, cheap_shots=256, cheap_repeats=1,
                                  promote_top_k=1, enable_hardware=False),
    score=rung_config.score, step_budget=2, patience=1, hardware=HardwareConfig(),
)

rung2 = RungConfig(
    rung=2, name="Rung 2", description="Refine optimization",
    objective="Tune optimization level",
    bootstrap_incumbent=ExperimentSpec(
        rung=2, target_backend="fake_brisbane", noise_backend="fake_brisbane",
        shots=256, repeats=1,
    ),
    search_space=SearchSpaceConfig(
        dimensions={"optimization_level": [1, 2, 3], "verification": ["both", "z_only"]},
        max_challengers_per_step=4,
    ),
    tier_policy=rung1.tier_policy,
    score=rung_config.score, step_budget=2, patience=1, hardware=HardwareConfig(),
)

results = harness3.run_ratchet([rung1, rung2], allow_hardware=False)

for lesson_obj, fb in results:
    print(f"\nRung {lesson_obj.rung}: {lesson_obj.name}")
    print(f"  Rules: {len(fb.rules)}")
    best_fields = {k: v for k, v in list(fb.best_spec_fields.items())[:4]}
    print(f"  Best spec: {best_fields}...")

print(f"\nAccumulated lessons: {len(harness3._accumulated_lessons)} rungs of feedback")

---
## 12. Transfer Evaluation

A transfer test checks if the best settings generalize across different backend noise profiles (not overfit to one).

In [ ]:
evaluator = TransferEvaluator()
report = evaluator.evaluate_across_backends(
    incumbent,
    ["fake_brisbane"],  # Use single backend for speed
    fast_rung,
)
print(f"Transfer score (pessimistic = min): {report.transfer_score:.4f}")
print(f"Mean score:  {report.mean_score:.4f}")
for name, score in report.per_backend_scores.items():
    print(f"  {name}: {score:.4f}")

In [ ]:
quiz(tracker, "q10_transfer",
    question="A spec scores 0.8 on one backend but 0.3 on another. What does this mean?",
    options=[
        "The spec is bad overall",
        "The spec is overfitted to the first backend's noise profile",
        "The second backend is broken",
    ],
    correct=1, section="12. Transfer", bloom="evaluate",
    explanation="A large transfer drop means settings are tuned to one backend's quirks. The ratchet tests transfer to find robust, generalizable configurations.")
checkpoint_summary(tracker, "12. Transfer")

---
## Summary

| Concept | What it does |
|---|---|
| **NeighborWalk** | Single-axis mutations — systematic but limited |
| **RandomCombo** | Multi-axis mutations — discovers interactions |
| **LessonGuided** | Rule-biased mutations — focuses on promising regions |
| **Ratchet step** | Generate, evaluate, promote, select winner |
| **Patience** | Stop early if no improvement |
| **Lesson extraction** | Human-readable + machine-readable rules from data |
| **Search narrowing** | Prune bad values, lock good ones |
| **Cross-rung propagation** | Winner and lessons flow to the next rung |
| **Transfer evaluation** | Check generalization across noise profiles |

> **Dashboard Exercise:** Try to manually find the best configuration in `00_dashboard.ipynb`. Then compare your best score to what the ratchet found. Who wins?

---
## Final Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")

---
## You've completed Plan C!

Want to explore the same material from a different angle? Try another plan:
- [Plan A — Sequential](../plan_a/01_encoded_magic_state.ipynb) (step-by-step, three notebooks)
- [Plan B — Spiral Notebook](../plan_b/spiral_notebook.ipynb) (three passes, increasing depth)
- [Plan D — Hypothesis-Driven](../plan_d/experiment_1_protection.ipynb) (experimental method)

*← [Dashboard](00_dashboard.ipynb) · [Track B — Engineering](track_b_engineering.ipynb) · [Start Here](../00_START_HERE.ipynb)*